# CIS I — Assignment 2: Kinematic Calibration of a Cartesian Robot

This notebook walks through the calibration scenario from the handout
(Figure 1): a cartesian-base + spherical-wrist robot carrying a special
calibration marker body, observed by an optical tracker. Your job is to
**design** the scene (where the tracker and robot go, what the calibration
body's marker layout is), **collect** data under a simulated clock, and
**fit** an improved kinematic model `ImprovedFwdKins` / `ImprovedInvKins`
that beats the nominal one over the robot's work volume: the handout asks
for `||alpha|| <= 1e-3 rad` and `||epsilon|| <= 0.2 mm`, in as little
simulated time as possible — and if you cannot get there, to report the
accuracy you achieved and how you went about it.

Everything runs against `src.simulation`, a self-contained simulator playing
the role of the physical robot + tracker (the tracker and marker-body
classes are assignment 1's, imported from `assignments/assignment1`). It
injects the kinematic errors and measurement noise — your code
never sees ground truth except where explicitly noted for
**validation/reporting** in Steps 8–10.

Unlike assignment 1, your code lives in Python modules under `src/student/`
(every function there is a stub raising `NotImplementedError`), driven from
this notebook and unit-tested from `tests/`. Keep the signatures the
notebook relies on; If you want to design the functions in a different way - feel free to do that. 
But indicate that in your report.
otherwise the design is yours. Remember the assignment's
closing note: this is a design exercise — part of the job is deciding which
items of the tracker's and robot's "spec sheet" matter for your error
budget, and your report must justify those decisions.

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "..")))

import numpy as np
from src import simulation, vct3, Rot, Frame
from src.simulation import clock, constants
from src.student import metrics, registration, workflow, kinematics

np.set_printoptions(precision=4, suppress=True)
clock.reset()

## Step 1: Design the scene

Specify approximate poses (as `Frame`) for the tracking camera and the robot
base, relative to the workspace, and the calibration body's marker layout
(`List[vct3]`, local coordinates). The simulator perturbs the robot's
placement the way the handout says (`||alpha_rob|| <= 0.05`,
`||epsilon_rob|| <= 5 mm`), so `F_rob_approx` is a design value, not truth.

Things to keep in mind:

- The tracker's usable volume (Figure 2) is a frustum along its **local
  +z axis**: 100–220 cm in front of the camera, widening from a 100 cm to a
  200 cm square. `trk.is_visible(points)` checks it for you.
- This handout's tracker has **no jiggle** (`sigma_Ta = sigma_Tp = 0`): the
  pose you give it is exactly where it is, so the tracker frame is a fixed
  reference and everything you measure is expressed in it. What remains is
  0.1 mm of random noise per marker reading and a small systematic field
  (`<= 0.1 mm`, slowly varying).
- The robot's end effector sits 600 mm along the base x-axis at home, and
  the linear joints travel `0..600, -300..300, 0..600` mm — so the work
  volume is a 600 mm cube spanning x ∈ [600, 1200], y ∈ [−300, 300],
  z ∈ [0, 600] in base coordinates, and the wrist (±0.75π, ±0.75π, ±π)
  swings the 75 mm calibration body around inside it. Everything must stay
  visible.
- The calibration body's layout is exact (`a*_k == a_k`) but limited to
  `||a*_k|| <= 75 mm`. Its orientation is estimated from its markers, so
  the number of markers and how far they spread set how noisy each
  orientation reading is.
- You may define and place additional marker bodies
  (`simulation.define_marker_body` / `place_marker_body`) if your design
  wants them; their layouts are only known to ±2 mm (see Step 5).

In [3]:
# TODO (DONE): Frame, tracker pose relative to the workspace (the camera looks along its local +z)
F_tracker_approx = Frame(Rot.eye(), vct3(900, 0, -1300))

# TODO (DONE): Frame, robot base pose relative to the workspace
# Made at origin for simplicity
F_rob_approx = Frame(Rot(), vct3(0, 0, 0))     

# TODO (DONE): points, every ||a*_k|| <= 75mm, for the calibration body on the end effector
# 74.48 mm ~= sqrt(43^2 * 3)
mbee_local_positions = [
    vct3( 43,  43,  43),
    vct3( 43, -43, -43),
    vct3(-43,  43, -43),
    vct3(-43, -43,  43)
]


## Step 2: Build the simulation

Fully provided — this wires your Step 1 choices through the simulator's
`CreateRobot` / `CreateTracker` / `CreateRobotCalibrationBody` /
`AddMarkerBodies` equivalents. The robot's hidden kinematic errors and the
tracker's hidden systematic-error field are drawn from `SEED`; your
calibration must work for *any* seed (grading uses different ones), so
don't tune to it.

`mbee_placed` is a `PlacedBody` handle pairing the calibration body with its
hidden actual placement; the body starts attached and moves with the robot.
Note that `rob.state()` already reports non-zero encoder values: the robot
starts at its physical home, `theta* = 0`, and the encoders read
`-delta_theta_sys(home) - delta_theta_rnd` there (see `rob.home_robot()`).

In [4]:
SEED = 1

rob = simulation.create_robot("rob", F_rob_approx, seed=SEED)
trk = simulation.create_tracker("trk", F_tracker_approx, seed=SEED)
mbee_placed = simulation.create_robot_calibration_body(rob, mbee_local_positions)
simulation.add_marker_bodies(trk, [mbee_placed])

print("simulation built:", rob, "|", trk.name, "| tracking", [p.body.name for p in trk.tracked_bodies])
print("robot state:", rob.state())

simulation built: Robot(name='rob', theta_cmd=[-0.0323, -0.0808, 0.0449, -0.0073, -0.0086, 0.0078]) | trk | tracking ['mBee']
robot state: RobotState(theta=array([-0.0323, -0.0808,  0.0449, -0.0073, -0.0086,  0.0078]), theta_dot=array([0., 0., 0., 0., 0., 0.]), t_elapsed=0.0, calibration_body_attached=True)


### Step 2.1: Visibility check

Before spending any simulated time, sweep a joint grid and check that
the calibration body stays inside the tracker's volume (using the *nominal*
placements — they're only tens of mm off, which is nothing against the
frustum size). If a large fraction is invisible, go back to Step 1.

In [5]:
def mbee_markers_world(theta):
    F_ee_world = rob.F_rob * rob.fwd_kins(theta)
    return [F_ee_world * a for a in mbee_local_positions]

n_vis, n_tot = 0, 0
for x in (0, 300, 600):
    for y in (-300, 0, 300):
        for z in (0, 300, 600):
            for w in (-1.8, 0.0, 1.8):
                theta = np.array([x, y, z, w, -w, w])
                n_tot += 1
                n_vis += trk.is_visible(mbee_markers_world(theta))
print(f"calibration body visible at {n_vis}/{n_tot} corner/edge poses of the work volume")

calibration body visible at 81/81 corner/edge poses of the work volume


## Step 3: Unit tests

`tests/test_simulation.py` is provided and passes as shipped — read it as
an example. `tests/test_student.py` exercises *your* `src/student` code:
the complete tests there fail with `NotImplementedError` until you
implement what they call, and the ones marked `TODO` are yours to write (you are required to write unit tests of each significant element, described in your
report together with their results). Re-run this cell as you go.

In [6]:
!{sys.executable} -m pytest ../tests -q -p no:cacheprovider

..............................s.s...s                                    [100%]
34 passed, 3 skipped in 0.29s


## Step 4: Error metric

You have to measure the mismatch between two frames as
`dF = F_a^-1 F_b ≈ Frame(I + sk(alpha), epsilon)`. Implement
`metrics.frame_error` (returning `||alpha||`, `||epsilon||`) and
`metrics.summarize_errors`; every accuracy number below goes through them.

In [7]:
print("sanity:", metrics.frame_error(Frame.eye(), Frame(Rot('z', 0.01), vct3(1, 0, 0))))

sanity: (0.01, 1.0)


## Step 5 (optional): Registration and marker-body refinement

> "It is possible to improve your estimate of a_m,k through a calibration
> process involving multiple observations of the marker body using the
> tracking camera and then averaging them."

The calibration body needs no refinement — `a*_k == a_k` by construction —
so this step only matters if your design places *additional* marker bodies
(their layouts are only known to ±2 mm, and the tracker registers each
body against its *current* nominal layout). You will need your own
`registration.register_points` (Arun's method — do not call the
simulator's solver) and `registration.average_frames` for Step 6 in any
case; `registration.refine_marker_body` is the optional part.

In [8]:
print("register_points sanity:", registration.register_points(
    [vct3(0, 0, 0), vct3(50, 0, 0), vct3(0, 50, 0), vct3(0, 0, 50)],
    [vct3(10, 0, 0), vct3(60, 0, 0), vct3(10, 50, 0), vct3(10, 0, 50)]).p)

register_points sanity: vct3(x=10.00, y=0.00, z=0.00)
Internal Vector:
[[10.]
 [ 0.]
 [ 0.]]


## Step 6: Home the robot, then a baseline in the tracker frame

> "`theta_encoders = HomeRobot()` will place the robot at a home position in
> which `theta*_k = 0` and return the current values read by the encoders.
> This may prove useful as part of your calibration process."

The encoders read `theta = theta* - delta_theta_rnd - delta_theta_sys`, so at
the physical home they expose the systematic offset of every joint
directly. Implement `workflow.measure_home_offsets` (average a few homings).

Then the baseline: the calibration body's coordinate system is exactly
`F_ee`, so every reading satisfies

    F_ee,j  =  F_X · FwdKins(theta_j)

for one constant unknown `F_X` (the robot base expressed in the tracker's
frame). Implement `workflow.collect_calibration_data` (robot move + one
`sample_marker_bodies` reading, storing the body's frame) and
`workflow.estimate_base_frame` (you'll want `registration.average_frames`),
then look at how well the **nominal** kinematics explain even a handful of
poses.

In [9]:
home_offsets = workflow.measure_home_offsets(rob, n_homings=5) # you can change the number of homings
print("delta_theta_sys at home (from the encoders):", home_offsets)

center = np.array([300.0, 0.0, 300.0, 0.0, 0.0, 0.0])
baseline_poses = [center] + [center + np.eye(6)[k] * v for k in range(3) for v in (-250.0, 250.0)] \
                 + [center + np.eye(6)[3] * 0.8, center + np.eye(6)[4] * 0.8, center + np.eye(6)[5] * 1.5]

baseline = workflow.collect_calibration_data(rob, trk, mbee_placed, baseline_poses, repeats=1)
F_X_baseline = workflow.estimate_base_frame(baseline, rob)
residuals = [metrics.frame_error(F_obs, F_X_baseline * rob.fwd_kins(th)) for th, F_obs in zip(baseline.thetas, baseline.frames)]
print(f"{len(baseline)} baseline samples in {baseline.t_collect:.1f}s")
print("nominal kinematics vs observations:", metrics.summarize_errors(residuals))

delta_theta_sys at home (from the encoders): [ 0.0299  0.0803 -0.0474  0.0074  0.0086 -0.0079]
10 baseline samples in 83.8s
nominal kinematics vs observations: {'n': 10, 'rms_angle': 0.008267832733858402, 'max_angle': 0.019482581913984137, 'rms_pos': 2.8980401539390974, 'max_pos': 5.01790252760088}


## Step 7: Plan and collect the calibration data

> "The goal is to reach a specified accuracy within as short a time as
> possible." — `clock.elapsed` is the score's time component.

The handout suggests three routes and *strongly* urges the first:

1. **Estimate the kinematic parameters.** (a) Work out how closely spaced
   your samples along an axis must be for the accuracy you want — this
   depends on the error bounds `delta_sys^trk`, `omega_sys^trk`,
   `delta_sys^rob`, `omega_sys^rob`; (b) home the robot, then systematically
   move along/about each individual axis while taking data, returning home
   after each axis; (c) determine the angles between the axes and the
   `theta -> theta*` map (use any interpolation scheme you find feasible); (d) write the corrected
   forward and inverse kinematics; (e) report the updated parameters and,
   in debugging, compare them with ground truth; then "bottom line" test a
   sampled portion of the workspace.
2. **Map the space and interpolate** — many more, closely spaced points,
   and dense random-point testing.
3. A mix.

Design `workflow.plan_calibration_poses`: which joint vectors to visit, in
which order. Every parameter of your model must be excited (axis tilts need
travel along each axis; wrist axes, `F_we` and the wrist joints' distortion
need wrist rotations; the linear joints' distortion needs stops along each
joint), every pose must keep the calibration body visible, and the wrist
moves at only 0.1 rad/s. Joints 1–3 are kinematically decoupled from 4–6.

`plan_calibration_poses` takes the accuracy you are aiming for and must
size the plan from it: how many distinct poses, and how many readings of
each (a pose listed twice in a row is re-commanded, costing only `t_settle`
— which also re-rolls the joint repeatability noise — and re-read for
`delta_t_samp`). With 0.1 mm of marker noise on a <= 75 mm body, how many
readings does a 1e-3 rad orientation estimate need? Work it out before you
spend simulated time.

In [10]:
DESIRED_ACCURACY = constants.TARGET_ANG_ERR_RAD  # rad; the orientation target is the limiting one

poses = workflow.plan_calibration_poses(rob, trk, mbee_local_positions, desired_accuracy=DESIRED_ACCURACY)  # TODO: your design config
n_distinct = len({tuple(np.round(th, 6)) for th in poses})
print(f"plan: {len(poses)} readings over {n_distinct} distinct poses")

t0 = clock.elapsed
data = workflow.collect_calibration_data(rob, trk, mbee_placed, poses, repeats=1)
print(f"{len(data)} samples collected in {data.t_collect:.1f}s; clock now {clock.elapsed:.1f}s")

F_X_nom = workflow.estimate_base_frame(data, rob)

Joint 1: 20/20 poses accepted
Joint 2: 20/20 poses accepted
Joint 3: 20/20 poses accepted
Joint 4: 20/20 poses accepted
Joint 5: 20/20 poses accepted
Joint 6: 20/20 poses accepted
plan: 120 readings over 119 distinct poses


120 samples collected in 326.2s; clock now 410.4s


## Step 8: Fit the improved kinematics

This is the assignment's core deliverable — implement
`kinematics.ImprovedKinematics` yourself and describe it in the report.

The handout's error model: `theta*_k = theta_k + delta_theta_k,rnd +
delta_theta_k,sys` on every joint (smooth, `||delta_theta_sys|| <=
delta_sys`, slope `<= omega_sys`), `d*_k = R(phi_k) d_k` for `k = 2..6`
(`d*_1 = d_1`), `F*_we = F_we dF_ee`, plus the base frame `F_X`. Choose a
parameter vector (the class docstring sketches one), write
`fwd_kins(theta, params)`, a residual against `data`, and a non-linear
least-squares `fit` (`scipy.optimize` is fine; so is a numerical
Jacobian). `inv_kins` is provided: it iterates on *your* `fwd_kins`.

**Questions for the report** (your `jacobian_rank_report` should help):
is every parameter identifiable from your poses? Is every parameter in the
handout's model identifiable *at all* — what happens to a component of
`phi_k` along `d_k`, to a rotation of the base about `d_1`, or to a joint
offset on joints 1–3 versus a shift of the base? (Which is why the home
offsets come from `HomeRobot()` rather than from the fit.) Which of the
handout's error terms are big enough to matter at the 0.2 mm / 1e-3 rad
level, and which are below the noise? How did that shape your
parameterization and your pose plan?

**Ground truth for debugging only**: `rob.ground_truth_parameters()`
returns the hidden kinematic parameters so you can compare your
`parameters_report()` with them, as the handout asks for debugging runs —
never use it in the calibration itself.

In [11]:
improved = kinematics.ImprovedKinematics(rob, F_X_nom, joint_offsets=home_offsets)  # TODO(DONEish): your constructor arguments
fit_info = improved.fit(data)
print("fit:", fit_info)

rank = improved.jacobian_rank_report(data)
print("jacobian rank:", rank["rank"], "/", rank["n_params"])

t_calibration = clock.elapsed
print(f"total simulated calibration time: {t_calibration:.1f}s (soft budget {constants.TARGET_TIME_SEC:.0f}s)")

estimated, truth = improved.parameters_report(), rob.ground_truth_parameters()   # truth: debugging only
print("estimated axis angles (deg):", estimated["axis_angles_deg"])
print("estimated joint offsets:", estimated["joint_offset"], "| true:", truth["joint_offset"])
print("estimated joint scales :", estimated["joint_scale"], "| true:", truth["joint_scale"])

# TODO(DONE): compare the rest (axes / tilts, F_we offset) -- mind the base-frame gauge
print("Fit succeeded:", fit_info.success)
print("Message:", fit_info.message)
print("Final cost:", fit_info.cost)

rank_report = improved.jacobian_rank_report(data)

print("Rank:", rank_report["rank"], "/", rank_report["n_params"])
print("Singular values:", rank_report["singular_values"])

fit:      message: `ftol` termination condition is satisfied.
     success: True
      status: 2
         fun: [-1.017e+00  1.201e+00 ... -4.993e-01 -2.728e-01]
           x: [ 5.630e-02 -2.038e-03 ... -2.917e-04  8.918e-04]
        cost: 126.42342111044358
         jac: [[-1.000e+03  6.859e+00 ... -0.000e+00  0.000e+00]
               [-6.769e+00 -9.995e+02 ... -0.000e+00  0.000e+00]
               ...
               [ 4.238e+00  9.448e+01 ... -5.006e+02 -5.006e+02]
               [-2.730e-01  1.998e+03 ...  5.224e+00  5.224e+00]]
        grad: [ 6.624e-03 -8.793e-02 ... -1.594e-04  1.421e-03]
  optimality: 0.09243126742967434
 active_mask: [ 0.000e+00  0.000e+00 ...  0.000e+00  0.000e+00]
        nfev: 18
        njev: 12
jacobian rank: 34 / 34
total simulated calibration time: 410.4s (soft budget 900s)
estimated axis angles (deg): [[ 0.     90.0639 90.1738  0.7238 91.1157 90.4119]
 [90.0639  0.     88.3105 90.7394  1.355  88.5217]
 [90.1738 88.3105  0.     89.9339 89.1618  0.3184]
 

## Step 9: Evaluate `ImprovedFwdKins` over the work volume

**Ground truth for reporting only**: `rob.actual_ee_pose(theta)` (the
actual end-effector pose in the workspace for an encoder command `theta`)
is a hidden value you have access to here only because this notebook
doesn't gate it. Use it *only* in Steps 9–10 to compute and report your
achieved error. With no tracker jiggle, the tracker's true pose is the one
you specified, so `F_trk = trk.nominal_pose()` maps between the workspace
and the tracker frame.

The metric, per test pose `theta_i`, with `F̂_X` your estimated base frame:

    dF_i = (F_trk^-1 · F*_ee(theta_i))^-1 · (F̂_X · ImprovedFwdKins(theta_i))

Report `||alpha||` and `||epsilon||` statistics over the grid below (which
is fixed — don't change it) for the nominal kinematics and for yours, and
compare against `constants.TARGET_POS_ERR_MM` / `TARGET_ANG_ERR_RAD`.
Evaluation reads the oracle only; it must not move the robot or sample the
tracker.

In [12]:
test_levels = {0: (100.0, 300.0, 500.0), 1: (-200.0, 0.0, 200.0), 2: (100.0, 300.0, 500.0),
               3: (-1.3, 0.0, 1.3), 4: (-1.3, 0.0, 1.3), 5: (-2.0, 0.0, 2.0)}
test_grid = [np.array([x, y, z, a, b, c])
             for x in test_levels[0] for y in test_levels[1] for z in test_levels[2]
             for a in test_levels[3] for b in test_levels[4] for c in test_levels[5]]
test_grid = [th for th in test_grid if trk.is_visible(mbee_markers_world(th))]
print(f"{len(test_grid)} visible test poses")

F_trk = trk.nominal_pose()

fwd_err_nominal, fwd_err_improved = [], []
for theta in test_grid:
    # TODO(DONE), for each test pose:
    #   1. truth = F_trk.inv() * rob.actual_ee_pose(theta)   (ground truth, reporting only)
    truth = F_trk.inv() * rob.actual_ee_pose(theta)
    #   2. prediction with the nominal kinematics: F_X_nom * rob.fwd_kins(theta)
    nominal_prediction = F_X_nom * rob.fwd_kins(theta)
    #   3. prediction with yours: improved.predict_in_tracker(theta)
    improved_prediction = improved.predict_in_tracker(theta)
    #   4. append metrics.frame_error(truth, prediction) to the two lists
    fwd_err_nominal.append(metrics.frame_error(truth, nominal_prediction))
    fwd_err_improved.append(metrics.frame_error(truth, improved_prediction))

print("Nominal:", metrics.summarize_errors(fwd_err_nominal))
print("Improved:", metrics.summarize_errors(fwd_err_improved))

729 visible test poses
Nominal: {'n': 729, 'rms_angle': 0.03272846606910886, 'max_angle': 0.0549756414163675, 'rms_pos': 7.94159059767002, 'max_pos': 16.857952281568966}
Improved: {'n': 729, 'rms_angle': 0.0005349992988660718, 'max_angle': 0.0008962038345897103, 'rms_pos': 0.10299557265354715, 'max_pos': 0.16979770070485925}


## Step 10: Evaluate `ImprovedInvKins`

For a target pose `F` (in the tracker frame), `theta =
ImprovedInvKins(F̂_X^-1 F)`, then actually move the robot there and compare
where it ends up — `mbee_placed.actual_placement` after the move, relative to
`F_trk` — with the target. Use targets you know are reachable (e.g.
ground-truth poses of test-grid joint vectors). The handout asks for the
time required for testing as well as for calibration, so keep track of it.
Note that every real move includes the joint repeatability noise
`sigma_rnd`; think about what floor that puts on this number.

In [13]:
rng = np.random.default_rng(0)
targets = [test_grid[i] for i in rng.choice(len(test_grid), size=min(40, len(test_grid)), replace=False)]
t_test_start = clock.elapsed

inv_err_nominal, inv_err_improved = [], []
for theta_t in targets:
    F_target = F_trk.inv() * rob.actual_ee_pose(theta_t)   # a reachable target, in the tracker frame
    # TODO (Done), for the nominal kinematics (rob, F_X_nom) and yours (improved, improved.base_frame()):
    for kins, F_X, errors in [(rob, F_X_nom, inv_err_nominal), (improved, improved.base_frame(), inv_err_improved),]:
        #   1. theta = kins.inv_kins(F_X.inv() * F_target, seed=theta_t)
        theta = kins.inv_kins(F_X.inv() * F_target, seed=theta_t)
        #   2. rob.move_joints(theta)
        rob.move_joints(theta)
        #   3. actual = F_trk.inv() * mbee_placed.actual_placement   (ground truth, reporting only)
        actual = F_trk.inv() * mbee_placed.actual_placement 
        #   4. append metrics.frame_error(F_target, actual)
        errors.append(metrics.frame_error(F_target, actual))

t_testing = clock.elapsed - t_test_start

print("Nominal inverse:", metrics.summarize_errors(inv_err_nominal))
print("Improved inverse:", metrics.summarize_errors(inv_err_improved))
print("Testing time:", t_testing)

Nominal inverse: {'n': 40, 'rms_angle': 0.034410494979572784, 'max_angle': 0.05480354665999439, 'rms_pos': 8.281132172106773, 'max_pos': 14.259971613286577}
Improved inverse: {'n': 40, 'rms_angle': 0.0005867539226129016, 'max_angle': 0.0010412473995264524, 'rms_pos': 0.10187461559159039, 'max_pos': 0.1600234699332614}
Testing time: 1089.2348403973888


## Achieved performance

Summarize: nominal vs improved, forward and inverse, against the handout's
targets (0.2 mm, 1e-3 rad), plus the simulated time spent calibrating and
testing and how the calibration time was spent (homing, number of
moves/samples). If a target is not met, say what you achieved and why —
that is an acceptable outcome per the handout. Put the same table in your
report, together with results for at least one other `SEED`.

In [14]:
# TODO: summarize fwd_err_*/inv_err_* with metrics.summarize_errors against
# constants.TARGET_POS_ERR_MM / TARGET_ANG_ERR_RAD, and report t_calibration and t_testing.
# Summarize forward and inverse kinematics errors
results = {
    "Forward (Nominal)": metrics.summarize_errors(fwd_err_nominal),
    "Forward (Improved)": metrics.summarize_errors(fwd_err_improved),
    "Inverse (Nominal)": metrics.summarize_errors(inv_err_nominal),
    "Inverse (Improved)": metrics.summarize_errors(inv_err_improved),
}

print("=== KINEMATICS ACCURACY ===")
print(f"Position error target:    {constants.TARGET_POS_ERR_MM} mm")
print(f"Orientation error target: {constants.TARGET_ANG_ERR_RAD} rad")
print()

for name, summary in results.items():
    print(f"--- {name} ---")
    for metric, value in summary.items():
        print(f"{metric}: {value}")
    print()

print("=== SIMULATED TIME ===")
print(f"Calibration time: {t_calibration:.2f} s")
print(f"Testing time:     {t_testing:.2f} s")
print(f"Total time:       {t_calibration + t_testing:.2f} s")

=== KINEMATICS ACCURACY ===
Position error target:    0.2 mm
Orientation error target: 0.001 rad

--- Forward (Nominal) ---
n: 729
rms_angle: 0.03272846606910886
max_angle: 0.0549756414163675
rms_pos: 7.94159059767002
max_pos: 16.857952281568966

--- Forward (Improved) ---
n: 729
rms_angle: 0.0005349992988660718
max_angle: 0.0008962038345897103
rms_pos: 0.10299557265354715
max_pos: 0.16979770070485925

--- Inverse (Nominal) ---
n: 40
rms_angle: 0.034410494979572784
max_angle: 0.05480354665999439
rms_pos: 8.281132172106773
max_pos: 14.259971613286577

--- Inverse (Improved) ---
n: 40
rms_angle: 0.0005867539226129016
max_angle: 0.0010412473995264524
rms_pos: 0.10187461559159039
max_pos: 0.1600234699332614

=== SIMULATED TIME ===
Calibration time: 410.39 s
Testing time:     1089.23 s
Total time:       1499.63 s
